# Nyaya — Eval-v1 + BhashaBench comparison run

Answers the question none of the published numbers could: **did the fine-tune help?**

Eval-v0's metric scored its own gold answers at ~10.7%, so base/v3/v4 were pinned
within a 2-answer spread by the ruler, not by the models. Eval-v1 has a 100% gold
ceiling, so a real difference can now show up.

**Before running:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**,
and add your HF read token as a Kaggle Secret named `HF_TOKEN`.

Runtime is roughly 60–90 min on one T4. Every run saves `predictions.jsonl`, so
scoring can be redone later on CPU without re-running any model.

In [ ]:
# --- setup -------------------------------------------------------------
!pip -q install -U transformers accelerate peft sentence-transformers rank_bm25 huggingface_hub

import os, subprocess, sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print("No HF_TOKEN secret found — gated datasets (BhashaBench) will fail:", exc)

REPO = "https://github.com/JitendraJha98/nyaya-model.git"
if not os.path.exists("/kaggle/working/nyaya-model"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/nyaya-model"], check=True)
os.chdir("/kaggle/working/nyaya-model")
sys.path.insert(0, "src")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# --- build Eval-v1 -----------------------------------------------------
# Only the PUBLIC half is committed to GitHub; the private half is gitignored.
# Regenerating from v0 here reproduces both halves deterministically.
!python scripts/25_build_eval_v1.py

import torch
print("CUDA:", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

## 1. Eval-v1: base vs the published release

Same retriever, same prompt, same questions — the only variable is the weights.
That is what makes the comparison meaningful.

In [ ]:
# Base model + dense RAG
!python scripts/26_eval_v1_run.py --adapter none --dense --k 8 \
    --split all --batch-size 8 --label base

In [ ]:
# The published merged release, pulled straight from the Hub
!python scripts/26_eval_v1_run.py --model NyayaLabs98/nyaya-3b-v3 --adapter none \
    --dense --k 8 --split all --batch-size 8 --label nyaya-3b-v3

In [ ]:
# Optional size-matched general baseline — is a legal fine-tune worth it at all?
!python scripts/26_eval_v1_run.py --model meta-llama/Llama-3.2-3B-Instruct \
    --adapter none --dense --k 8 --split all --batch-size 8 --label llama-3.2-3b

## 2. BhashaBench-Legal — external, MCQ-scored

MCQ scoring is exact, so there is no phrase-matching problem and the numbers are
directly comparable against any other model. This is the benchmark to quote publicly.

In [ ]:
import json, re, torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

SUBSET = 1500  # keep the run tractable; raise once the pipeline is proven

bench = load_dataset("bharatgenai/BhashaBench-Legal", split="test")
if SUBSET and len(bench) > SUBSET:
    bench = bench.shuffle(seed=0).select(range(SUBSET))
print(len(bench), "questions |", bench.column_names)


def mcq_prompt(row):
    opts = "\n".join(f"{k}. {row[k]}" for k in ("A", "B", "C", "D")
                     if row.get(k) not in (None, ""))
    return (f"{row['question']}\n{opts}\n\n"
            "Answer with the single letter of the correct option.")


def run_mcq(model_id, label, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=torch.bfloat16, device_map="auto").eval()

    correct, total, rows = 0, 0, []
    for start in range(0, len(bench), batch_size):
        batch = bench.select(range(start, min(start + batch_size, len(bench))))
        texts = [tok.apply_chat_template(
            [{"role": "user", "content": mcq_prompt(r)}],
            tokenize=False, add_generation_prompt=True) for r in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=8, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        decoded = tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True)
        for row, text in zip(batch, decoded):
            match = re.search(r"\b([ABCD])\b", text.upper())
            pred = match.group(1) if match else None
            gold = str(row["answer"]).strip().upper()[:1]
            correct += int(pred == gold)
            total += 1
            rows.append({"pred": pred, "gold": gold, "raw": text})
        print(f"\r{label}: {total}/{len(bench)} acc={correct/max(1,total):.1%}",
              end="", flush=True)

    del model
    torch.cuda.empty_cache()
    with open(f"/kaggle/working/bhashabench_{label}.jsonl", "w") as fh:
        for r in rows:
            fh.write(json.dumps(r) + "\n")
    print(f"\n{label}: {correct}/{total} = {correct/total:.1%}")
    return correct / total


scores = {}
for model_id, label in [("Qwen/Qwen2.5-3B-Instruct", "base"),
                        ("NyayaLabs98/nyaya-3b-v3", "nyaya-3b-v3")]:
    scores[label] = run_mcq(model_id, label)

print(json.dumps(scores, indent=2))

## 3. Results — download these

`predictions.jsonl` files matter most: they let scoring be revised later without a GPU.
That is exactly what was missing when the v0 scorer turned out to be broken.

In [ ]:
import json, shutil, pathlib

results = json.load(open("reports/eval_v1_results.json"))
print(f"{'run':<18} {'fact_recall':>12} {'citation':>10} {'all_facts':>10}")
for label, payload in results.items():
    m = payload["metrics"]
    print(f"{label:<18} {m['fact_recall']:>11.1%} {m['citation_accuracy']:>10.1%} "
          f"{m['all_facts_accuracy']:>10.1%}")

out = pathlib.Path("/kaggle/working/nyaya-eval-v1-results")
out.mkdir(exist_ok=True)
shutil.copy("reports/eval_v1_results.json", out)
for run in pathlib.Path("outputs/eval-v1").glob("*/predictions.jsonl"):
    shutil.copy(run, out / f"{run.parent.name}_predictions.jsonl")
shutil.make_archive("/kaggle/working/nyaya-eval-v1-results", "zip", out)
print("\nDownload /kaggle/working/nyaya-eval-v1-results.zip from the Output tab.")